# 14c — LPT spherical density at 32³: MCLMC (pixel + harmonic)

Unadjusted microcanonical Langevin (BlackJAX MCLMC via `batched_sampling`) on the joint `(Ω_c, σ_8, δ_IC)` posterior for the pixel and the harmonic density likelihood. MCLMC costs ~1 gradient per step (vs ~10–100 for a NUTS draw), so it is the cheap field sampler — at the price of tuning (`desired_energy_var`) and thinning to decorrelate.

Same config + mock (seed 0) as `14a-LPTDensityMUSE` and `14b-LPTDensityNUTS`. Run headless with `uv run --no-sync papermill 14c-LPTDensityMCLMC.ipynb 14c-LPTDensityMCLMC.ipynb --cwd .`.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

os.environ["JAX_ENABLE_X64"] = "True"  # float32 => NaN/chaotic IC gradients
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")  # coexist with other GPU processes
# At 32^3 the pipeline is a chain of small sequential ops: a many-core CPU can beat a small GPU
# (measured: 0.14 s/grad on a 20-core CPU vs 0.51 s on an RTX 4060). Set JAX_PLATFORMS=cpu to force CPU.
os.environ.setdefault("JAX_PLATFORMS", "cuda,cpu")

import dataclasses
import time

import jax
import jax.numpy as jnp
import jax_cosmo as jc
import matplotlib.pyplot as plt
import numpy as np
import jax_fli as jfli
from jax.scipy.special import ndtr
from numpyro.handlers import condition, seed, trace

jax.config.update("jax_enable_x64", True)

MESH = 32  # laptop/cluster-node size; bump for production or use the distributed 15-lensing-muse-inference.py
NSIDE = MESH
print(f"jax {jax.__version__}  backend {jax.default_backend()}  x64 {jax.config.jax_enable_x64}  MESH={MESH}")

## 1. Model configuration and mock observation

LPT-only (`sim_mode="lpt"`) → spherical galaxy overdensity (`lensing_output="density"`), two tomographic source planes at z = 0.10 and 0.17 inside the z ≤ 0.2 lightcone, centered observer (whole sky). The mock is a forward draw of the pixel model at a random prior point (seed 0 — identical across the 14a/14b/14c notebooks, so their posteriors are directly comparable); we condition on it and warm-start at the truth.

In [ ]:
cosmo = jc.Planck18()
box = tuple(float(x) for x in jfli.utils.compute_box_size_from_redshift(cosmo, 0.2, (0.5, 0.5, 0.5)))
priors = {
    "Omega_c": jfli.infer.PreconditionnedUniform(0.1, 0.5),
    "sigma8": jfli.infer.PreconditionnedUniform(0.6, 1.0),
}

config = jfli.ppl.Configurations(
    mesh_size=(MESH, MESH, MESH),
    box_size=box,
    halo_size=(0, 0),
    field_sharding=None,
    sim_mode="lpt",
    nbody_solver="BullFrog",
    t0=0.001,
    t1=1.0,
    lpt_order=1,
    number_of_shells=5,
    nb_steps=5,
    paint_order="cic",
    gradient_order=4,
    laplace_fd=True,
    shell_spacing="a",
    time_stepping="D",
    min_width=1.0,
    lensing_output="density",
    map2alm_method="jax",
    likelihood_space="pixel",
    min_redshift=0.001,
    max_redshift=0.2,
    n_integrate=8,
    nside=NSIDE,
    geometry="spherical",
    scheme="rbf_neighbor",
    observer_position=(0.5, 0.5, 0.5),
    paint_nside=NSIDE,
    kernel_width_pixels=0.8,
    fiducial_cosmology=jc.Planck18,
    nz_shear=[0.10, 0.17],
    priors=priors,
    sigma_e=0.3,
    adjoint="checkpointed",
    checkpoints=2,
)

pixel_model = jfli.ppl.full_field_probmodel(config)
tr = trace(seed(pixel_model, 0)).get_trace()
x_obs = jnp.stack([v["value"] for k, v in tr.items() if "observable" in k and k != "observable_meta_data"], axis=0)
x_obs_meta_data = tr["observable_meta_data"]["value"]
theta_truth = jnp.array([float(tr["Omega_c_base"]["value"]), float(tr["sigma8_base"]["value"])])
Oc_true, s8_true = float(tr["Omega_c"]["value"]), float(tr["sigma8"]["value"])
truth = {"Omega_c": Oc_true, "sigma8": s8_true}
print(f"truth: Omega_c={Oc_true:.4f}  sigma8={s8_true:.4f}   observable {x_obs.shape}")

# Harmonic variant of the same config: NEVER mutate the shared dataclass in place — use dataclasses.replace.
ell_max, taper = min(2 * NSIDE - 1, 3 * NSIDE // 2), 4
config_h = dataclasses.replace(config, likelihood_space="harmonic", ell_max=int(ell_max), ell_taper_width=taper)
harmonic_model = jfli.ppl.full_field_probmodel(config_h, observed_maps=x_obs)

In [ ]:
jfli.SphericalDensity.FromDensityMetadata(array=x_obs, field=x_obs_meta_data).show()

## 2. Sanity gate: jitted log-density + gradient at the truth

Expect **large** cosmology-base gradients (O(10²–10³) at 32³): at the true parameters the score is a zero-mean random variable with std = √Fisher, and a field-level likelihood has a big Fisher information for 2 parameters. This is expected statistics, not a normalization bug (no noise variance depends on the sampled cosmology) — see `docs/WORK_IN_PROGRESS/20-likelihood-audit.md`. Time gradients **under `jax.jit`**: an eager `jax.grad` call is dispatch-bound and ~50× slower at this size (the historical "8 s per gradient" was that artifact).

In [ ]:
data = {f"observable_{i}": x_obs[i] for i in range(x_obs.shape[0])}
cond_model = condition(pixel_model, data=data)

from numpyro.infer.util import initialize_model

init, potential_fn, postprocess_fn, model_trace = initialize_model(jax.random.key(0), cond_model)
logdensity_fn = lambda position: -potential_fn(position)
value_and_grad_fn = jax.jit(jax.value_and_grad(logdensity_fn))

truth_position = {
    "Omega_c_base": theta_truth[0],
    "sigma8_base": theta_truth[1],
    "initial_conditions": tr["initial_conditions"]["value"].array,
}
t0 = time.time()
val, grad = jax.block_until_ready(value_and_grad_fn(truth_position))
print(f"compile+first-run: {time.time() - t0:.1f} s")
t0 = time.time()
for _ in range(3):
    val, grad = jax.block_until_ready(value_and_grad_fn(truth_position))
print(f"jitted gradient: {(time.time() - t0) / 3 * 1e3:.0f} ms")
assert np.isfinite(float(val)) and all(bool(np.all(np.isfinite(np.asarray(g)))) for g in jax.tree.leaves(grad))
print(f"logp={float(val):.1f}  dOc_base={float(grad['Omega_c_base']):.3e}  ds8_base={float(grad['sigma8_base']):.3e}")

## 3. MCLMC: pixel, then harmonic

Field-inference settings: `desired_energy_var=1e-7` (the 1e-3 default is for small models), `diagonal_preconditioning=True`, and `init_step_size_scale=1e-4` — a plain scalar; `batched_sampling` multiplies it by √dim itself (passing √dim·scale here double-counts and blows up the first step). `thinning=3` stores every 3rd step so consecutive draws decorrelate.

In [ ]:
t0 = time.time()
jfli.infer.batched_sampling(
    cond_model,
    path="output/nb14c/mclmc_pixel",
    rng_key=jax.random.PRNGKey(0),
    num_warmup=2000,
    num_samples=2000,
    batch_count=1,
    sampler="MCLMC",
    thinning=3,
    mclmc_desired_energy_var=1e-7,
    mclmc_init_step_size_scale=1e-4,
    mclmc_diagonal_preconditioning=True,
    init_params=truth_position,
    progress_bar=False,
    save_callback=jfli.infer.sample2catalog(config),
    post_process=lambda s: {
        **s,
        "initial_conditions": jfli.interpolate_initial_conditions(
            s["initial_conditions"], config.mesh_size, config.box_size, cosmo=s["cosmo"]
        ).array,
    },
)
print(f"MCLMC pixel done in {time.time() - t0:.0f}s   (see output/nb14c/mclmc_pixel/metrics.md)")

In [ ]:
t0 = time.time()
jfli.infer.batched_sampling(
    harmonic_model,  # data already bound via observed_maps (observed harmonic_obs site) -> no condition()
    path="output/nb14c/mclmc_harmonic",
    rng_key=jax.random.PRNGKey(1),
    num_warmup=2000,
    num_samples=2000,
    batch_count=1,
    sampler="MCLMC",
    thinning=3,
    mclmc_desired_energy_var=1e-7,
    mclmc_init_step_size_scale=1e-4,
    mclmc_diagonal_preconditioning=True,
    init_params=truth_position,
    progress_bar=False,
    save_callback=jfli.infer.sample2catalog(config_h),
    post_process=lambda s: {
        **s,
        "initial_conditions": jfli.interpolate_initial_conditions(
            s["initial_conditions"], config.mesh_size, config.box_size, cosmo=s["cosmo"]
        ).array,
    },
)
print(f"MCLMC harmonic done in {time.time() - t0:.0f}s   (see output/nb14c/mclmc_harmonic/metrics.md)")

In [ ]:
extracts = {
    "MCLMC-pixel": jfli.io.extract_catalog(
        set_name="MCLMC-pixel", cosmo_keys=["Omega_c", "sigma8"], patterns=["output/nb14c/mclmc_pixel/samples"]
    ),
    "MCLMC-harmonic": jfli.io.extract_catalog(
        set_name="MCLMC-harmonic", cosmo_keys=["Omega_c", "sigma8"], patterns=["output/nb14c/mclmc_harmonic/samples"]
    ),
}
# tag the first extract with the truth so plot_posterior draws the truth markers
ex0 = extracts["MCLMC-pixel"]
extracts["MCLMC-pixel"] = jfli.io.CatalogExtract(name=ex0.name, cosmo=ex0.cosmo, truth_cosmo=truth)

In [ ]:
S = {name: {k: (ex.cosmo[k].ravel().mean(), ex.cosmo[k].ravel().std()) for k in ("Omega_c", "sigma8")} for name, ex in extracts.items()}
for name, s in S.items():
    print(
        f"{name:16s}  Omega_c={s['Omega_c'][0]:.3f}+/-{s['Omega_c'][1]:.3f}   sigma8={s['sigma8'][0]:.3f}+/-{s['sigma8'][1]:.3f}"
    )
print(f"{'truth':16s}  Omega_c={truth['Omega_c']:.3f}           sigma8={truth['sigma8']:.3f}")

for name, s in S.items():
    for k in ("Omega_c", "sigma8"):
        m, sd = s[k]
        assert abs(m - truth[k]) < 3.0 * max(sd, 1e-6), f"{name} {k}={m:.3f} off truth {truth[k]:.3f} (sigma {sd:.3f})"
print("PASS: every run recovers the true cosmology within 3 sigma of its own spread")

jfli.infer.plot_posterior(list(extracts.values()), labels={"Omega_c": r"\Omega_c", "sigma8": r"\sigma_8"})
plt.show()

## Methods note

MCLMC is unadjusted (no Metropolis correction): correctness relies on the energy-error control (`desired_energy_var`), so treat a drifting or NaN-flagged run (`metrics.md` reports the NaN-free fraction) as a tuning failure, not a posterior. For an adjusted variant at similar cost, `batched_sampling(sampler="MAMS")` runs the Metropolis-adjusted microcanonical sampler. Compare against 14b (NUTS) on ESS per gradient: a NUTS draw costs ~2^tree-depth gradients, an MCLMC step costs ~1.